In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from urllib.request import urlretrieve

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
urlretrieve(data, 'course_lead_scoring_2026.csv')

('course_lead_scoring_2026.csv', <http.client.HTTPMessage at 0x7f06cd363ed0>)

In [3]:
df  = pd.read_csv('course_lead_scoring_2026.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [16]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [17]:

# Fill missing values
for col in categorical:
    df[col] = df[col].fillna('NA')
for col in numerical:
    df[col] = df[col].fillna(0.0)

In [19]:
df.industry.mode()

0    technology
Name: industry, dtype: object

### Question 2 – Highest correlation among numerical features

In [20]:
corr = df[numerical].corr()
print(corr)

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f'{a} vs {b}: {corr.loc[a, b]:.4f}')

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  
interaction_count vs lead_score: 0.9157
number_of_courses_viewed vs lead_score: 0.7572
number_of_courses_viewed vs interaction_count: 0.7216
annual_income vs interaction_count: 0.1228


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

In [22]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3

In [23]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(calculate_mi).sort_values(ascending=False)
print(mi.round(2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


### Question 4

In [24]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(round(acc, 2))

0.65


### Question 5

In [25]:
features = categorical + numerical
orig_acc = accuracy_score(y_val, model.predict(X_val))

def train_and_acc(feats):
    dv_local = DictVectorizer(sparse=False)
    X_tr = dv_local.fit_transform(df_train[feats].to_dict(orient='records'))
    X_v = dv_local.transform(df_val[feats].to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    return accuracy_score(y_val, m.predict(X_v))

for feat in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    feats_without = [f for f in features if f != feat]
    acc_without = train_and_acc(feats_without)
    diff = orig_acc - acc_without
    print(f'{feat}: acc_without={acc_without:.6f}, difference={diff:.6f}')

lead_source: acc_without=0.642000, difference=0.003000
number_of_courses_viewed: acc_without=0.643000, difference=0.002000
interaction_count: acc_without=0.601000, difference=0.044000


### Question 6

In [26]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    a = accuracy_score(y_val, m.predict(X_val))
    print(f'C={C}: {round(a, 3)}')

C=1e-06: 0.598
C=1e-05: 0.598
C=0.0001: 0.613
C=0.001: 0.645
